### Medical Appointment Data Cleaning Pipeline

### Import Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/")

### Data ingestion and schema

In [0]:
df = spark.read.format('csv').option("inferSchema", True).option("header", True).load("/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df_schema = StructType([
    StructField("PatientID", IntegerType()),
    StructField("AppointmentID", IntegerType()),
    StructField("Gender", StringType()),
    StructField("ScheduledDay", TimestampType()),
    StructField("AppointmentDay", TimestampType()),
    StructField("Age", IntegerType()),
    StructField("Neighbourhood", StringType()),
    StructField("Scholarship", IntegerType()),
    StructField("Hipertension", IntegerType()),
    StructField("Diabetes", DoubleType()),
    StructField("Alcoholism", IntegerType()),
    StructField("Handcap", IntegerType()),
    StructField("SMS_received", IntegerType()),
    StructField("No-show", StringType()),
])

In [0]:
df = spark.read.format('csv').option("inferSchema", True).option("header", True).load("/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv")
df.display()

In [0]:
df_custom_schema = spark.read.format('csv').schema(df_schema).option("header", True).load("/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv")

In [0]:
print("INFERRED SCHEMA")
df.printSchema()

print("CUSTOM SCHEMA")
df_custom_schema.printSchema()

### Selecting and renaming columns

In [0]:
df = df.withColumnRenamed("PatientId", "patient_id") \
       .withColumnRenamed("AppointmentID", "appointment_id") \
       .withColumnRenamed("Gender", "gender") \
       .withColumnRenamed("ScheduledDay", "scheduled_day") \
       .withColumnRenamed("AppointmentDay", "appointment_day") \
       .withColumnRenamed("Age", "age") \
       .withColumnRenamed("Neighbourhood", "neighbourhood") \
       .withColumnRenamed("Scholarship", "scholarship") \
       .withColumnRenamed("Hipertension", "hypertension") \
       .withColumnRenamed("Diabetes", "diabetes") \
       .withColumnRenamed("Alcoholism", "alcoholism") \
       .withColumnRenamed("Handcap", "handicap") \
       .withColumnRenamed("SMS_received", "sms_received") \
       .withColumnRenamed("No-show", "no_show")

In [0]:
df_alias = df.select(col("patient_id")).alias("patient_identifier")

In [0]:
df_alias.display()

### Filtering and investigating records

#### Scenario 1

In [0]:
df.filter(col("age") < 0).display()

#### Scenario 2


In [0]:
df.filter(col("age") < 18).display()

In [0]:
df.filter(col("no_show")=="Yes").display()

In [0]:
df.filter((col("sms_received") == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("scholarship") == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("hypertension") == 1) & (col('diabetes') == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("age").isNull()) | col("neighbourhood").isNull())
df.display()

### Creating and modifying columns

In [0]:
df = df.withColumn("data_source", lit("Kaggle Medical Dataset"))
df.display()

In [0]:
df = df.withColumn("appointment_status",when(col("no_show") == "Yes", "No-show").otherwise("Show"))
df.display()

In [0]:
df = df.withColumn("gender", upper(col("gender")))
df.display()

In [0]:
df = df.withColumn("age_group", when((col("age")) <18, "Under 18").when((col("age")) <36, "18-35").when((col("age")) <51, "36-50").when((col("age")) <66, "51-65").otherwise("Above 65"))
df.display() 

In [0]:
df.withColumnRenamed("no_show", "no_show_flag")
df.display()


### Type casting and sorting

In [0]:
df.select(
    "scheduled_day",
    "appointment_day"
).show(10, truncate=False)

In [0]:
df.withColumn('scheduled_day', col('scheduled_day').cast(TimestampType())).display()

In [0]:

df.withColumn('appointment_day', col('appointment_day').cast(TimestampType())).display()

In [0]:
df.withColumn('age', col('age').cast(IntegerType())).display()

In [0]:
df.withColumn('hypertension', col('hypertension').cast(BooleanType())).display()
df.withColumn('diabetes', col('diabetes').cast(BooleanType())).display()
df.withColumn('alcoholism', col('alcoholism').cast(BooleanType())).display()
df.withColumn('handicap', col('handicap').cast(BooleanType())).display()

In [0]:
df.sort(col("age").desc()).display()

In [0]:
df.sort(["appointment_day", "age"], ascending = [1, 0]).display()

In [0]:
df.limit(20).display()

In [0]:
df.display(20)

In [0]:
df.limit(20).display()

In [0]:
df.limit(100).display()

### Removing columns and duplicates


In [0]:
df.drop("patient_id").display()

In [0]:
df.drop("alcoholism", "scholarship").display()

In [0]:
df.groupBy(*df.columns).count().filter(col("count")>1).display()

In [0]:
df.dropDuplicates().display()

In [0]:
df.groupBy("appointment_id").count().filter(col("count")>1).display()

In [0]:
df.groupBy("patient_id").count().filter(col("count")>1).display()